## Step1:Importing Packages

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from delta.tables import *
from pyspark.sql import SparkSession

**Insight**:

Imports the required   and libraries and packages.

## Step2:Create Spark Session

In [0]:
sess=SparkSession\
    .builder\
    .appName("deltatable")\
        .getOrCreate()

**Insight**:

Creating a Spark session to perform spark processing.

## Step3:Load The Dataset

In [0]:
data=sess.read.format("csv")\
    .option("inferSchema","true")\
        .option("header","true")\
            .load("/Volumes/dbacademy/default/data/retail_orders.csv")

**Insight**:

Reads a CSV retail dataset into a PySpark DataFrame .

## Step4:Exploration Of Dataset

In [0]:
display(data.limit(20))

order_id,customer_name,gender,age,city,state,order_date,product,category,quantity,unit_price,payment_mode,delivery_status,rating,returned
1,Connie Boyd,Male,41,Delhi,Delhi,24/11/2023,Phone,Electronics,2,599,null,Delivered,-1,false
2,Daniel Jimenez,Male,30,Delhi,Karnataka,01-17-2023,Laptop,Electronics,-2,599,Crypto,Pending,-1,false
3,Mark Young,F,41,Jaipur,Delhi,2023-04-23,Headphones,Accessories,0,29999,Cash,Pending,4,false
4,Richard Smith,M,22,Mumbai,Maharashtra,22/02/2023,Phone,Electronics,-2,29999,UPI,null,-1,true
5,Marcus Wang,Male,150,Mumbai,Karnataka,05-13-2024,Printer,Electronics,3,1499,null,Delivered,3,null
6,Brianna Dennis,Male,30,Jaipur,Gujarat,2023-04-27,Monitor,Electronics,3,-500,null,Cancelled,4,false
7,Jose Mitchell,Unknown,22,Bengaluru,Delhi,01/07/2023,Keyboard,Accessories,-2,1499,Card,null,8,false
8,Nathan Marshall,Unknown,22,Delhi,Rajasthan,04-18-2024,Keyboard,Accessories,3,-500,Cash,Delivered,4,null
9,Diana Richardson,Female,-5,Ahmedabad,Gujarat,2024-03-24,Tablet,Electronics,2,29999,Card,Pending,-1,null
10,Tiffany James DDS,F,-5,Mumbai,Delhi,15/05/2023,Printer,Electronics,2,-500,UPI,Delivered,3,true


**Insight**:

Display dataset records.

In [0]:
print(f"The shape of the dataset is :({len(data.columns)},{data.count()})")
print(f"No of rows:{data.count()}")
print(f"No of columns:{len(data.columns)}")

The shape of the dataset is :(15,1100)
No of rows:1100
No of columns:15


**Insight**:

Shape of the Dataset.

In [0]:
data.columns

['order_id',
 'customer_name',
 'gender',
 'age',
 'city',
 'state',
 'order_date',
 'product',
 'category',
 'quantity',
 'unit_price',
 'payment_mode',
 'delivery_status',
 'rating',
 'returned']

**Insight**:

List of columns of sample Dataset.

In [0]:
data.printSchema()

root
 |-- order_id: integer (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- order_date: string (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: integer (nullable = true)
 |-- payment_mode: string (nullable = true)
 |-- delivery_status: string (nullable = true)
 |-- rating: integer (nullable = true)
 |-- returned: boolean (nullable = true)



**Insight**:

Prints the schema to check data types of columns.

## Step5:Data Cleaning

#### 5.1:Handling Null Values

In [0]:
null_values = data.select([
    count(when(col(x).isNull(), x)).alias(x)
    for x in data.columns
])

display(null_values)

order_id,customer_name,gender,age,city,state,order_date,product,category,quantity,unit_price,payment_mode,delivery_status,rating,returned
0,98,0,79,36,42,0,30,27,0,0,368,288,59,380


**Insight**:

Identifies missing values in each column.

#### 5.1.1:Droping null values

In [0]:
clean_data=data.dropna(subset="customer_name")

**Insight**:

Removing rows with null values in column(customer_name) 

#### 5.1.2:Filling null Values

In [0]:
avg_age=clean_data.select("age").agg(avg("age")).collect()[0][0]
avg_rating=clean_data.select("rating").agg(avg("rating")).collect()[0][0]
clean_data=clean_data.fillna({
    "age":avg_age,
    "rating":avg_rating,
    "unit_price":0,
    "quantity":0,
    "city":"unknown",
    "state":"unknown",
    "product":"unknown",
    "category":"unknown",
    "payment_mode":"unknown",
    "delivery_status":"unknown"
})

**Insight**:

Filling missing values with appropriate values

#### 5.1.3:Handling column values

In [0]:
clean_data=clean_data.withColumn("unit_price", when(col("unit_price")<0,-1*col("unit_price"))\
    .otherwise(col("unit_price")))
clean_data=clean_data.withColumn("quantity",when(col("quantity")<0,lit(0))\
    .otherwise(col("quantity")))

clean_data=clean_data.withColumn("rating",when(col("rating")<0,lit(0))\
    .otherwise(col("rating")))



**Insight**:

Correct invalid values of columns.

In [0]:
null_values = clean_data.select([
    count(when(col(x).isNull(), x)).alias(x)
    for x in clean_data.columns
])

display(null_values)

order_id,customer_name,gender,age,city,state,order_date,product,category,quantity,unit_price,payment_mode,delivery_status,rating,returned
0,0,0,0,0,0,0,0,0,0,0,0,0,0,347


**Insight**:

Checking null values after handling missing values

### 5.2:Handling Duplicate Records

#### 5.2.1:No of records before Removing Duplicates.

In [0]:
data.count()

1100

In [0]:
clean_data=clean_data.dropDuplicates(["order_id"])

#### 5.2.2:No of records after removing duplicates.

In [0]:
clean_data.count()

868

**Insight**:

Removing Duplicate records from the dataset.

In [0]:
display(clean_data.limit(20))

order_id,customer_name,gender,age,city,state,order_date,product,category,quantity,unit_price,payment_mode,delivery_status,rating,returned
12,Debbie Bailey,F,41,Mumbai,Rajasthan,2023-12-15,Phone,Electronics,3,500,Card,unknown,3,null
18,Michael Jenkins,F,30,Jaipur,Rajasthan,2023-10-11,Keyboard,Accessories,1,1499,UPI,Delivered,5,true
38,John Holt,Male,41,Delhi,Karnataka,04-25-2024,Laptop,Electronics,3,599,UPI,unknown,3,false
67,Ann Riley,Female,30,Ahmedabad,Maharashtra,16/01/2024,Mouse,Accessories,0,29999,UPI,Pending,5,true
70,Julie Benitez,Male,22,Ahmedabad,Gujarat,10/05/2023,Headphones,Accessories,1,599,Card,Pending,0,false
93,Jennifer Moore,Female,30,unknown,Rajasthan,2024-02-10,Mouse,Accessories,0,500,Crypto,Pending,8,null
161,Manuel Weeks,F,30,Bengaluru,Maharashtra,08-16-2023,Phone,Electronics,0,29999,unknown,Cancelled,4,true
186,Eric Simmons,M,30,unknown,Gujarat,2023-08-14,Tablet,Electronics,2,500,Crypto,unknown,3,null
190,James Phillips,M,41,Delhi,Maharashtra,21/10/2023,Printer,Electronics,1,29999,unknown,Delivered,3,false
218,Arthur Pham,Female,150,Delhi,Gujarat,01-22-2024,Headphones,Accessories,2,29999,Cash,unknown,8,null


#### 5.3:Sort the Dataset

In [0]:
clean_data=clean_data.sort(col("order_id"))

**Insight**:

Sort the dataset by order_id.

## Step6:Working with Delta Lake

#### 6.1:Save the Dataset as a Delta Table

In [0]:
clean_data.write.format("delta")\
.mode("overwrite")\
.saveAsTable("retail_table")

**Insight**:

Saves the cleaned DataFrame as a Delta table (retail_table).

#### 6.2:Display the Delta Table

In [0]:
%sql
select *from retail_table
limit 5

order_id,customer_name,gender,age,city,state,order_date,product,category,quantity,unit_price,payment_mode,delivery_status,rating,returned
1,Connie Boyd,Male,41,Delhi,Delhi,24/11/2023,Phone,Electronics,2,599,unknown,Delivered,0,false
2,Daniel Jimenez,Male,30,Delhi,Karnataka,01-17-2023,Laptop,Electronics,0,599,Crypto,Pending,0,false
3,Mark Young,F,41,Jaipur,Delhi,2023-04-23,Headphones,Accessories,0,29999,Cash,Pending,4,false
4,Richard Smith,M,22,Mumbai,Maharashtra,22/02/2023,Phone,Electronics,0,29999,UPI,unknown,0,true
5,Marcus Wang,Male,150,Mumbai,Karnataka,05-13-2024,Printer,Electronics,3,1499,unknown,Delivered,3,null


**Insight**:

Display the delta table records.

#### 6.3:Read the Source CSV File

In [0]:
source_data=spark.read.format("csv")\
    .option("header","true")\
        .option("inferSchema","true")\
            .load("/Volumes/dbacademy/default/data/source_file.csv")

**Insight**:

Reads an incremental/source CSV dataset.

#### 6.4:Display the Source data

In [0]:
display(source_data)

order_id,customer_name,gender,age,city,state,order_date,product,category,quantity,unit_price,payment_mode,delivery_status,rating,returned
101,Rahul Sharma,Male,30,Jaipur,Rajasthan,2026-07-01,Laptop,Electronics,2,62000,UPI,Delivered,5,false
350,Priya Verma,Female,27,Delhi,Delhi,2026-07-02,Headphones,Electronics,1,2500,Credit Card,Delivered,4,false
720,Amit Singh,Male,35,Lucknow,Uttar Pradesh,2026-07-03,Office Chair,Furniture,1,8500,Cash on Delivery,Shipped,5,false
951,Neha Gupta,Female,29,Mumbai,Maharashtra,2026-07-04,Smartwatch,Electronics,1,12000,Debit Card,Processing,4,false
952,Karan Mehta,Male,40,Pune,Maharashtra,2026-07-04,Microwave Oven,Home Appliances,1,15000,UPI,Processing,5,false


**Insight**:

Display the Source Dataset/Incremental Dataset.

#### 6.5:Create a Reference to the Target Delta Table

In [0]:
target_table=DeltaTable.forName(sess,"retail_table")


**Insight**:

Creates a object of the  Delta table.

#### 6.6:Perform an Upsert (Merge) 

In [0]:
target_table.alias("tt").merge(
    source_data.alias("ss"),
    col("tt.order_id") ==  col("ss.order_id")                                    
).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

**Insight**:Performs an upsert (MERGE) operation:

-->Updates existing records when order_id matches.

-->Inserts new records when no matching order_id exists.

#### Step7:Validate the Merge Results


##### 7.1:No of Records before Merge Operation

In [0]:
clean_data.count()

868

#####7.2:No of Records after merge operation

In [0]:
spark.sql("select count(*) from retail_table").collect()[0][0]

870

##### 7.3:Duplicate records check

In [0]:
spark.sql("""
SELECT order_id,
COUNT(*)
FROM retail_table
GROUP BY order_id
HAVING COUNT(*)>1
""").show()

+--------+--------+
|order_id|COUNT(*)|
+--------+--------+
+--------+--------+



**Insight**

Check the duplicate records after the Incremental Load.

## Step8:Final Summary

#### 8.1:Final Delta Table

In [0]:
%sql
select * from retail_table
order by order_id

order_id,customer_name,gender,age,city,state,order_date,product,category,quantity,unit_price,payment_mode,delivery_status,rating,returned
1,Connie Boyd,Male,41,Delhi,Delhi,24/11/2023,Phone,Electronics,2,599,unknown,Delivered,0,false
2,Daniel Jimenez,Male,30,Delhi,Karnataka,01-17-2023,Laptop,Electronics,0,599,Crypto,Pending,0,false
3,Mark Young,F,41,Jaipur,Delhi,2023-04-23,Headphones,Accessories,0,29999,Cash,Pending,4,false
4,Richard Smith,M,22,Mumbai,Maharashtra,22/02/2023,Phone,Electronics,0,29999,UPI,unknown,0,true
5,Marcus Wang,Male,150,Mumbai,Karnataka,05-13-2024,Printer,Electronics,3,1499,unknown,Delivered,3,null
6,Brianna Dennis,Male,30,Jaipur,Gujarat,2023-04-27,Monitor,Electronics,3,500,unknown,Cancelled,4,false
7,Jose Mitchell,Unknown,22,Bengaluru,Delhi,01/07/2023,Keyboard,Accessories,0,1499,Card,unknown,8,false
8,Nathan Marshall,Unknown,22,Delhi,Rajasthan,04-18-2024,Keyboard,Accessories,3,500,Cash,Delivered,4,null
9,Diana Richardson,Female,-5,Ahmedabad,Gujarat,2024-03-24,Tablet,Electronics,2,29999,Card,Pending,0,null
10,Tiffany James DDS,F,-5,Mumbai,Delhi,15/05/2023,Printer,Electronics,2,500,UPI,Delivered,3,true


**Insight**

Merge operation successfully completed on the Delta table with the following results:

New Records Inserted: 951, 952

Existing Records Updated: 101, 720, 350


#### 8.2:Row count

In [0]:
print("Original records :", clean_data.count())

print("Incremental records :", source_data.count())

print("Final records :", spark.table("retail_table").count())

Original records : 868
Incremental records : 5
Final records : 870
